# KIE4033 Assignment 1: Exploratory Data Analysis of Safe Drinking Water Access (SDG 6)

**Data sources**
- WHO/UNICEF Joint Monitoring Programme for Water Supply, Sanitation and Hygiene (2025), processed by Our World in Data: safely managed drinking water and at least basic sanitation, by country, 2000-2024
- World Bank regions, via Our World in Data

**Research questions**
- RQ1: Does the share of people using safely managed drinking water differ between World Bank regions in 2024? (one-way ANOVA + Tukey HSD post-hoc)
- RQ2: Is access in Sub-Saharan Africa different from the rest of the world in 2024? (Wilcoxon rank sum test)
- Extra: how strongly is safe water access related to basic sanitation access? (correlation)

**How to run:** Runtime > Run all. Figures are saved in the `figures/` folder.

## 0. Setup

In [ ]:
# ---- Setup: all imports in one place ----
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# folder where the figures for the report are saved
os.makedirs("figures", exist_ok=True)

# Our World in Data asks for a user agent when downloading
UA = {"User-Agent": "Our World In Data data fetch/1.0"}
OWID = "https://ourworldindata.org/grapher/"
SUFFIX = ".csv?v=1&csvType=full&useColumnShortNames=false"


def save_and_show(filename):
    # save the current figure for the report, then display it
    plt.tight_layout()
    plt.savefig("figures/" + filename, dpi=150, bbox_inches="tight")
    plt.show()

## 1. Data understanding

In [ ]:
# download safely managed drinking water data (WHO/UNICEF JMP via Our World in Data)
df = pd.read_csv(OWID + "proportion-using-safely-managed-drinking-water" + SUFFIX, storage_options=UA)

print(df.shape)
print(df.columns.tolist())
print(df.head())

# missing values and years covered
print(df.isna().sum())
print("places:", df["Entity"].nunique(), "| years:", df["Year"].min(), "to", df["Year"].max())

## 2. Data cleaning

In [ ]:
# Remove aggregate rows (World, income groups, WHO and UN regions).
# They have no normal 3-letter country code, and keeping them would count the same people twice.
print("places before:", df["Entity"].nunique())
df_clean = df[df["Code"].str.match(r"^[A-Z]{3}$")].copy()
print("places after:", df_clean["Entity"].nunique())
print("rows:", len(df), "->", len(df_clean))

# short column name, to make later code easier to read
df_clean = df_clean.rename(
    columns={"Share of the population using safely managed drinking water": "safe_water_pct"}
)

# Some country-years are not in the table at all. We do NOT fill or guess them.
n_years = df_clean["Year"].nunique()
counts = df_clean["Entity"].value_counts()
print("expected rows if complete:", df_clean["Entity"].nunique() * n_years, "| actual rows:", len(df_clean))
print(counts[counts < n_years])

In [ ]:
# add the World Bank region of each country (merge on country code)
regions = pd.read_csv(OWID + "world-regions-according-to-the-world-bank" + SUFFIX, storage_options=UA)
regions_small = regions[["Code", "World regions according to WB"]].rename(
    columns={"World regions according to WB": "region"}
)
df_merged = df_clean.merge(regions_small, on="Code", how="left")

# checks: same number of rows, no country without a region
print(df_merged.shape)
print("rows with no region:", df_merged["region"].isna().sum())
print(df_merged.drop_duplicates("Code")["region"].value_counts())

In [ ]:
# add a second indicator: share of people using at least basic sanitation
# (the odd spelling "faciltities" is how the website spells it)
sanitation = pd.read_csv(
    OWID + "share-of-population-with-improved-sanitation-faciltities" + SUFFIX, storage_options=UA
)

# same cleaning rule as before: keep only real countries
print("places before:", sanitation["Entity"].nunique())
san_clean = sanitation[sanitation["Code"].str.match(r"^[A-Z]{3}$")].copy()
print("places after:", san_clean["Entity"].nunique())
san_clean = san_clean.rename(
    columns={"Share of the population using at least basic sanitation": "basic_san_pct"}
)

# left merge on country AND year: no water rows are lost, missing sanitation values stay blank (not filled)
df_all = df_merged.merge(san_clean[["Code", "Year", "basic_san_pct"]], on=["Code", "Year"], how="left")
print(df_all.shape)
print(df_all.isna().sum())

## 3. Summary statistics

In [ ]:
# both columns are percentages, so min and max should be between 0 and 100
print(df_all[["safe_water_pct", "basic_san_pct"]].describe())

# the 2024 subset used for the regional comparisons
df_2024 = df_all[df_all["Year"] == 2024]
print("countries with a 2024 value:", len(df_2024))
print(df_2024["region"].value_counts())

## 4. Exploratory data analysis

In [ ]:
# Distribution of all country-years: strongly left-skewed (many near 100%)
plt.figure(figsize=(8, 5))
plt.hist(df_clean["safe_water_pct"], bins=20, edgecolor="black")
plt.xlabel("Population using safely managed drinking water (%)")
plt.ylabel("Number of country-years")
plt.title("Distribution of safely managed drinking water access (2000-2024)")
save_and_show("01_histogram.png")

In [ ]:
# Average access over time (simple average across countries)
yearly_mean = df_clean.groupby("Year")["safe_water_pct"].mean()
print(yearly_mean.iloc[[0, -1]])

plt.figure(figsize=(8, 5))
plt.plot(yearly_mean.index, yearly_mean.values, marker="o")
plt.xlabel("Year")
plt.ylabel("Average population using safely managed drinking water (%)")
plt.title("Average safe drinking water access over time (2000-2024)")
plt.grid(True)
save_and_show("02_trend_average.png")

In [ ]:
# Lowest countries in 2024 and how many are below 50%
print(df_2024.sort_values("safe_water_pct").head(10)[["Entity", "safe_water_pct"]])
print((df_2024["safe_water_pct"] < 50).sum(), "out of", len(df_2024), "countries below 50%")

# Spread in three years: start, middle and end
df_years = df_clean[df_clean["Year"].isin([2000, 2012, 2024])]
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_years, x="Year", y="safe_water_pct")
plt.xlabel("Year")
plt.ylabel("Population using safely managed drinking water (%)")
plt.title("Safe drinking water access across countries (2000, 2012, 2024)")
save_and_show("03_boxplot_years.png")

In [ ]:
# Access by region in 2024, lowest median at the top
order = df_2024.groupby("region")["safe_water_pct"].median().sort_values().index

plt.figure(figsize=(10, 6))
sns.boxplot(data=df_2024, x="safe_water_pct", y="region", order=order)
plt.xlabel("Population using safely managed drinking water (%)")
plt.ylabel("World Bank region")
plt.title("Safe drinking water access by region (2024)")
save_and_show("04_boxplot_regions_2024.png")

In [ ]:
# Trend by region (simple average of the countries with data in each year)
region_trend = df_all.groupby(["Year", "region"])["safe_water_pct"].mean().reset_index()

plt.figure(figsize=(10, 6))
sns.lineplot(data=region_trend, x="Year", y="safe_water_pct", hue="region", marker="o")
plt.xlabel("Year")
plt.ylabel("Average population using safely managed drinking water (%)")
plt.title("Safe drinking water access by region over time (2000-2024)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
save_and_show("05_trend_by_region.png")

In [ ]:
# Check the dip in North America around 2005-2007: which countries have data each year?
na = df_all[(df_all["region"] == "North America (WB)") & (df_all["Year"].between(2003, 2010))]
print(na.pivot(index="Year", columns="Entity", values="safe_water_pct").round(1))

In [ ]:
# Correlation between safe water and basic sanitation, 2024 only (one row per country)
df_corr = df_2024.dropna(subset=["safe_water_pct", "basic_san_pct"])
print("countries used:", len(df_corr))
print(df_corr[["safe_water_pct", "basic_san_pct"]].corr())                    # Pearson
print(df_corr[["safe_water_pct", "basic_san_pct"]].corr(method="spearman"))   # Spearman (rank based)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_corr, x="basic_san_pct", y="safe_water_pct", hue="region")
plt.xlabel("Population using at least basic sanitation (%)")
plt.ylabel("Population using safely managed drinking water (%)")
plt.title("Safe water vs basic sanitation by country (2024)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
save_and_show("06_scatter_water_sanitation.png")

## 5. Statistical analysis

### RQ1: do regions differ in 2024? (one-way ANOVA + Tukey HSD)
- H0: the mean safe water access is the same in all regions
- H1: at least one region has a different mean

Assumptions are checked first (Levene for equal variance, Shapiro-Wilk for normality in each region).
A p-value below 0.05 means the assumption fails.

In [ ]:
# one list of 2024 values per region
groups = {name: g["safe_water_pct"].values for name, g in df_2024.groupby("region")}

# assumption 1: equal variance between regions
lev_stat, lev_p = stats.levene(*groups.values())
print("Levene test p-value:", lev_p)

# assumption 2: normality inside each region (n=3 and n=5 are too small to mean much)
for name, values in groups.items():
    stat, p = stats.shapiro(values)
    print(f"{name}: n={len(values)}, Shapiro p-value={p:.4f}")

In [ ]:
# one-way ANOVA
f_stat, p_val = stats.f_oneway(*groups.values())
df_between = len(groups) - 1
df_within = len(df_2024) - len(groups)
print(f"F({df_between}, {df_within}) = {f_stat:.2f}, p = {p_val:.3e}")

In [ ]:
# Tukey HSD post-hoc: which pairs of regions differ? (reject = True means significant at 5%)
tukey = pairwise_tukeyhsd(endog=df_2024["safe_water_pct"], groups=df_2024["region"], alpha=0.05)
print(tukey)

### RQ2: Sub-Saharan Africa vs the rest of the world in 2024 (Wilcoxon rank sum test)
- H0: the two groups have the same distribution
- H1: the distributions differ

Note: Sub-Saharan Africa was chosen after looking at the plots, which should be mentioned as a limitation.

In [ ]:
ssa = df_2024[df_2024["region"] == "Sub-Saharan Africa (WB)"]["safe_water_pct"]
rest = df_2024[df_2024["region"] != "Sub-Saharan Africa (WB)"]["safe_water_pct"]

print("n SSA:", len(ssa), "| n rest:", len(rest))
print("median SSA:", round(ssa.median(), 1), "| median rest:", round(rest.median(), 1))

# Wilcoxon rank sum test (called Mann-Whitney U in scipy)
stat, p = stats.mannwhitneyu(ssa, rest, alternative="two-sided")
print("U statistic:", stat)
print(f"p-value: {p:.3e}")

## References
- WHO/UNICEF Joint Monitoring Programme for Water Supply, Sanitation and Hygiene (2025). Estimates for drinking water, sanitation and hygiene services by country (2000-2024). https://washdata.org/data. Processed by Our World in Data.
- World Bank regions, processed by Our World in Data.